# sandbox_polars

In [1]:
import json
import polars as pl
from pathlib import Path
ROOT_PATH = Path.cwd().resolve().parent

def load_processed(file_name):
    '''
   Renvoie la dataframe polars à partir des données dont le nom (avec son extension) est donné en entrée.
    '''
    path_target_file = ROOT_PATH / "data" / "processed" / file_name
    return pl.read_parquet(path_target_file)

# Chargement des fichiers dans des dataframes polars
table_poi_df = load_processed("2026-07-22_143436_paris_poi.parquet")
table_connections_df = load_processed("2026-07-22_143436_connections.parquet")

print(table_poi_df.shape)
print(table_connections_df.shape)

(50, 9)
(99, 10)


# Exercice 1 — Créer et inspecter

Charge fichier_list (ta table poi) dans un DataFrame Polars. <br> Affiche ses dimensions, ses colonnes, et ses types de données.<br> 
Question à te poser : est-ce que les types inférés par Polars correspondent à ce que tu attendais ? (latitude/longitude en flottant, poi_id en entier...)

In [2]:
# premières lignes, comme Pandas
table_poi_df.head()        

poi_id,title,town,postcode,latitude,longitude,number_of_points,usage_cost,date_last_confirmed
i64,str,str,str,f64,f64,i64,str,str
7008,"""angle rue Poulet - Barbès""","""Paris""","""""",48.856614,2.352222,1,"""Free""","""2011-10-12T20:41:00Z"""
6931,"""Parking Lobau-Rivoli""","""Paris""",""" 75004""",48.856014,2.353281,1,"""Free""","""2011-10-10T18:18:00Z"""
60368,"""Place Saint-Gervais""","""Paris""","""75004""",48.855837,2.354095,3,""">3kW: 0,25€/15min <60min; 2,00…",null
6977,"""Place Saint-Gervais autolib""","""Paris""",null,48.855797,2.354148,1,"""Free""","""2011-10-12T19:39:00Z"""
7025,"""Parking Hôtel de Ville""","""Paris""","""75004""",48.856984,2.349716,2,"""Free""","""2011-10-18T09:15:00Z"""


In [3]:
# (nb_lignes, nb_colonnes), identique
table_poi_df.shape

(50, 9)

In [4]:
 # liste des noms de colonnes, identique
table_poi_df.columns

['poi_id',
 'title',
 'town',
 'postcode',
 'latitude',
 'longitude',
 'number_of_points',
 'usage_cost',
 'date_last_confirmed']

In [5]:
# types de chaque colonne
table_poi_df.dtypes

[Int64, String, String, String, Float64, Float64, Int64, String, String]

In [6]:
# affichage complet, formaté différemment de Pandas mais lisible
print(table_poi_df) 

shape: (50, 9)
┌────────┬─────────────┬───────┬──────────┬───┬───────────┬─────────────┬─────────────┬────────────┐
│ poi_id ┆ title       ┆ town  ┆ postcode ┆ … ┆ longitude ┆ number_of_p ┆ usage_cost  ┆ date_last_ │
│ ---    ┆ ---         ┆ ---   ┆ ---      ┆   ┆ ---       ┆ oints       ┆ ---         ┆ confirmed  │
│ i64    ┆ str         ┆ str   ┆ str      ┆   ┆ f64       ┆ ---         ┆ str         ┆ ---        │
│        ┆             ┆       ┆          ┆   ┆           ┆ i64         ┆             ┆ str        │
╞════════╪═════════════╪═══════╪══════════╪═══╪═══════════╪═════════════╪═════════════╪════════════╡
│ 7008   ┆ angle rue   ┆ Paris ┆          ┆ … ┆ 2.352222  ┆ 1           ┆ Free        ┆ 2011-10-12 │
│        ┆ Poulet -    ┆       ┆          ┆   ┆           ┆             ┆             ┆ T20:41:00Z │
│        ┆ Barbès      ┆       ┆          ┆   ┆           ┆             ┆             ┆            │
│ 6931   ┆ Parking Lob ┆ Paris ┆  75004   ┆ … ┆ 2.353281  ┆ 1           ┆ Fr

# Exercice 2 — Filtrer

Sur ton DataFrame poi, filtre pour ne garder que les POI dont number_of_points est supérieur à 1 (plusieurs points de charge sur le même site). <br>
Vérifie combien de lignes ça te donne par rapport au total.

In [7]:
extract = table_poi_df.filter(pl.col("number_of_points") > 1)
print(f"Il y a {extract.shape[0]} poi ayant plusieurs points de charge sur le même site.")
print(f"Cela représente {extract.shape[0]/table_poi_df.shape[0]*100} % du total.")

Il y a 11 poi ayant plusieurs points de charge sur le même site.
Cela représente 22.0 % du total.


# Exercice 3 — Sélectionner des colonnes précises

Depuis ton DataFrame **connections**, sélectionne uniquement les colonnes **connection_type**, **power_kw**, **is_fast_charge_capable** — sans les autres.

In [15]:
table_connections_df.select(pl.col("connection_type", "power_kw", "is_fast_charge_capable"))

connection_type,power_kw,is_fast_charge_capable
str,i64,bool
"""CCS (Type 2)""",22,true
"""CHAdeMO""",22,true
"""Type 2 (Socket Only)""",22,false
"""SCAME Type 3C (Schneider-Legra…",22,false
"""CEE 7/5""",3,false
…,…,…
"""Type 2 (Socket Only)""",7,false
"""CEE 7/4 - Schuko - Type F""",7,false
"""Type 2 (Socket Only)""",7,false


In [14]:
table_connections_df.select(["connection_type", "power_kw", "is_fast_charge_capable"])

connection_type,power_kw,is_fast_charge_capable
str,i64,bool
"""CCS (Type 2)""",22,true
"""CHAdeMO""",22,true
"""Type 2 (Socket Only)""",22,false
"""SCAME Type 3C (Schneider-Legra…",22,false
"""CEE 7/5""",3,false
…,…,…
"""Type 2 (Socket Only)""",7,false
"""CEE 7/4 - Schuko - Type F""",7,false
"""Type 2 (Socket Only)""",7,false


# Exercice 4 — Compter des occurrences (group_by)

Sur **connections**, regroupe par **connection_type** et compte combien de connecteurs existent pour chaque type. <br> 
Résultat attendu : une sorte de tableau de fréquence, un peu comme ton Counter de tout à l'heure, mais via Polars et SQL-like plutôt qu'une boucle Python.

In [9]:
table_connections_df.head()

poi_id,connection_id,power_kw,amps,voltage,connection_type,current_type,is_operational,level_title,is_fast_charge_capable
i64,i64,i64,i64,i64,str,str,bool,str,bool
60368,76450,22,120,400,"""CCS (Type 2)""","""DC""",true,"""Level 3: High (Over 40kW)""",true
60368,76451,22,120,400,"""CHAdeMO""","""DC""",true,"""Level 3: High (Over 40kW)""",true
60368,76452,22,32,400,"""Type 2 (Socket Only)""","""AC (Three-Phase)""",true,"""Level 2 : Medium (Over 2kW)""",false
60368,76453,22,32,400,"""SCAME Type 3C (Schneider-Legra…","""AC (Three-Phase)""",true,"""Level 2 : Medium (Over 2kW)""",false
60368,76454,3,10,16,"""CEE 7/5""","""AC (Single-Phase)""",true,"""Level 1 : Low (Under 2kW)""",false


In [10]:
table_connections_df.group_by("connection_type").agg(pl.col("connection_id").count())

connection_type,connection_id
str,u32
"""Type 2 (Tethered Connector) """,11
"""CCS (Type 2)""",11
"""CHAdeMO""",11
"""SCAME Type 3C (Schneider-Legra…",8
"""CEE 7/5""",6
"""CEE 7/4 - Schuko - Type F""",28
"""Type 2 (Socket Only)""",24


# Exercice 5 — Agréger avec une statistique

Toujours sur **connections**, regroupe par **connection_type**, mais cette fois calcule la puissance moyenne (**power_kw**) par type de connecteur plutôt qu'un simple comptage.<br>
Question à anticiper : que se passe-t-il si **power_kw** est **None** sur certaines lignes — Polars les ignore-t-il automatiquement dans le calcul de la moyenne, ou faut-il les traiter avant ?

In [11]:
table_connections_df.group_by("connection_type").agg(pl.col("power_kw").mean())

connection_type,power_kw
str,f64
"""CEE 7/4 - Schuko - Type F""",9.535714
"""SCAME Type 3C (Schneider-Legra…",17.25
"""CHAdeMO""",18.545455
"""Type 2 (Tethered Connector) """,21.0
"""CCS (Type 2)""",22.0
"""Type 2 (Socket Only)""",7.458333
"""CEE 7/5""",3.0


In [12]:
table_connections_df.select(pl.col("power_kw").is_null().sum())

power_kw
u32
0


# Exercice 6 — Jointure (le concept clé pour ton architecture à deux tables)

Fais une jointure entre **poi** et **connections** sur la colonne **poi_id**, pour obtenir un DataFrame combiné où chaque ligne de connexion affiche aussi la town du POI associé.<br>
C'est l'exercice le plus important des six : c'est exactement ce que tu feras plus tard en SQL sur DuckDB pour croiser localisation et caractéristiques techniques des bornes.

In [13]:
resultat = table_connections_df.join(other = table_poi_df.select(["poi_id","town"]), on="poi_id")
resultat

poi_id,connection_id,power_kw,amps,voltage,connection_type,current_type,is_operational,level_title,is_fast_charge_capable,town
i64,i64,i64,i64,i64,str,str,bool,str,bool,str
60368,76450,22,120,400,"""CCS (Type 2)""","""DC""",true,"""Level 3: High (Over 40kW)""",true,"""Paris"""
60368,76451,22,120,400,"""CHAdeMO""","""DC""",true,"""Level 3: High (Over 40kW)""",true,"""Paris"""
60368,76452,22,32,400,"""Type 2 (Socket Only)""","""AC (Three-Phase)""",true,"""Level 2 : Medium (Over 2kW)""",false,"""Paris"""
60368,76453,22,32,400,"""SCAME Type 3C (Schneider-Legra…","""AC (Three-Phase)""",true,"""Level 2 : Medium (Over 2kW)""",false,"""Paris"""
60368,76454,3,10,16,"""CEE 7/5""","""AC (Single-Phase)""",true,"""Level 1 : Low (Under 2kW)""",false,"""Paris"""
…,…,…,…,…,…,…,…,…,…,…
198895,331932,7,null,null,"""Type 2 (Socket Only)""","""AC (Single-Phase)""",true,"""Level 2 : Medium (Over 2kW)""",false,null
198885,331912,7,null,null,"""CEE 7/4 - Schuko - Type F""","""AC (Single-Phase)""",true,"""Level 2 : Medium (Over 2kW)""",false,null
198885,331913,7,null,null,"""Type 2 (Socket Only)""","""AC (Single-Phase)""",true,"""Level 2 : Medium (Over 2kW)""",false,null
